# 02. Limpeza e Tratamento de Dados (Data Cleaning & Normalization)

## Objetivos do Pipeline ETL
1. Corrigir o deslocamento de colunas (`rating` -> `duration`).
2. Validar consistência `type` vs `duration` antes de qualquer extração numérica.
3. Tratar imputações de nulos (`country`, `rating`).
4. Converter tipos de dados (`date_added` para datetime e extrair ano/mês).
5. Gerar datasets normalizados (títulos únicos, países explodidos, gêneros explodidos).
6. Validar cada etapa antes de exportar — nenhuma suposição sem checagem.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\lukin\OneDrive\Documentos\netflix_catalog_analysis\data\netflix_titles.csv')
print(f"Carga inicial: {len(df)} linhas")

mask_shift = df['rating'].str.contains('min', na=False)
print(f"Linhas com deslocamento detectado: {mask_shift.sum()} (esperando: 3)")
assert mask_shift.sum() == 3, "Numero de linhas deslocadas mudou - revisar"

df.loc[mask_shift, 'duration'] = df.loc[mask_shift, 'rating']
df.loc[mask_shift, 'rating'] = 'NR'

Carga inicial: 8807 linhas
Linhas com deslocamento detectado: 3 (esperando: 3)


In [4]:
#Checagem de Integridade (TYPE vs DURATION)

movie_duration_check = df[df['type'] == 'Movie']['duration'].str.contains('min', na=False).all()
tv_duration_check = df[df['type'] == 'TV Show']['duration'].str.contains('Season', na=False).all()

print(f"[Checagem de Integridade]")
print(f"--- Todos os Filmes contem 'min'? {movie_duration_check}")
print(f"--- Todas as Series contem 'Season'? {tv_duration_check}")

assert movie_duration_check, "Existe Filme sem 'min' em duration' - investigar antes de seguir"
assert tv_duration_check, "Existe Serie sem 'Season' em duration' - investigar antes de segurir"

[Checagem de Integridade]
--- Todos os Filmes contem 'min'? True
--- Todas as Series contem 'Season'? True


In [7]:
# Imputação de Nulos e Datas

#--- Imputação de Nulos ---
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('NR')

#--- Parsing de Datas ---
df['date_added'] = df['date_added'].str.strip()
df['date_added_clean'] = pd.to_datetime(df['date_added'], format= '%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added_clean'].dt.year
df['month_added'] = df['date_added_clean'].dt.month

nat_count = df['date_added_clean'].isnull().sum()
print(f"NaT em date_added_clean: {nat_count} (esperando: 10)")
assert nat_count == 10, "Divergencia no parsing de datas - investigar formato das linhas restantes..."

NaT em date_added_clean: 10 (esperando: 10)


In [ ]:
# --- Extração Numérica da Duração (raw string para evitar warning)
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)

dur_nulls = df['duration_num'].isnull().sum()
print(f"Nulos em duration_num: {dur_nulls} (esperado: 0)")
assert dur_nulls == 0, "Falha na extração numérica de duration - revisar linhas remanescentes"

Nulos em duration_num: 0 (esperado: 0)


In [ ]:
# --- Normalização para Explode (Paises e Generos)

df_countries = (
    df[['show_id', 'type', 'release_year', 'year_added', 'country']]
    .assign(country=df['country'].str.split(','))
    .explode('country')
)
df_countries['country'] = df_countries['country'].str.strip()

empty_countries = (df_countries['country'] =='').sum()
print(f"Strings vazias em country (pós-trip): {empty_countries}")
if empty_countries > 0: 
    df_countries = df_countries[df_countries['country'] !=''].copy()
    print(f"---{empty_countries} linhas vazias removidas de df_countries")

df_genres = (
    df[['show_id', 'type', 'release_year', 'year_added', 'listed_in']]
    .assign(genre=df['listed_in'].str.split(','))
    .explode('genre')
)
df_genres['genre'] = df_genres['genre'].str.strip()
df_genres = df_genres.drop(columns=['listed_in'])

empty_genres = (df_genres['genre'] == '').sum()
print(f"Strings vazias em genre (pós-strip): {empty_genres}")
if empty_genres > 0:
    df_genres = df_genres[df_genres['genre'] != ''].copy()
    print(f"-> {empty_genres} linha(s) vazia(s) removida(s) de df_genres.")

print(f"\nLinhas em df_genres: {len(df_genres)}")
print(f"Gêneros únicos: {df_genres['genre'].nunique()} (esperado: ~42)")
print(f"Média de gêneros por título: {len(df_genres) / df['show_id'].nunique():.2f} (esperado: entre 1.5 e 3.5)")
print(df_genres['genre'].head(10).tolist())

Strings vazias em country (pós-trip): 7
---7 linhas vazias removidas de df_countries
Strings vazias em genre (pós-strip): 0

Linhas em df_genres: 19323
Gêneros únicos: 42 (esperado: ~42)
Média de gêneros por título: 2.19 (esperado: entre 1.5 e 3.5)
['Documentaries', 'International TV Shows', 'TV Dramas', 'TV Mysteries', 'Crime TV Shows', 'International TV Shows', 'TV Action & Adventure', 'Docuseries', 'Reality TV', 'International TV Shows']


In [20]:
print("\n[Validação Pós-Limpeza]")
print(f"Base Principal (df): {len(df)} registros.")
print(f"Linhas em df_countries (explodido): {len(df_countries)}")
print(f"Países Únicos (pós-strip): {df_countries['country'].nunique()} (incluindo 'Unknown')")
print(f"Linhas em df_genres (explodido): {len(df_genres)}")
print(f"Gêneros Únicos (pós-strip): {df_genres['genre'].nunique()}")

assert len(df_countries) >= len(df), "df_countries tem menos linhas que títulos - erro no explode"
assert len(df_genres) >= len(df), "df_genres tem menos linhas que títulos - erro no explode"


[Validação Pós-Limpeza]
Base Principal (df): 8807 registros.
Linhas em df_countries (explodido): 10843
Países Únicos (pós-strip): 123 (incluindo 'Unknown')
Linhas em df_genres (explodido): 263791
Gêneros Únicos (pós-strip): 42


## Nota Metodológica — H2 (Expansão Internacional)

O valor `'Unknown'` atribuído aos 831 registros sem país conhecido (9,43%) é mantido em
`df_countries` para preservar a integridade da base. **No cálculo específico da métrica de H2**
(% de aparições fora dos EUA), os registros com `country == 'Unknown'` devem ser filtrados
tanto do numerador quanto do denominador — a ser aplicado no notebook `04_eda_regionalizacao.ipynb`,
não nesta etapa de limpeza.

In [25]:
import os

processed_dir = r'C:\Users\lukin\OneDrive\Documentos\netflix_catalog_analysis\data\processed'
os.makedirs(processed_dir, exist_ok=True)

df.to_csv(os.path.join(processed_dir, 'netflix_titles_clean.csv'), index=False)
df_countries.to_csv(os.path.join(processed_dir, 'netflix_countries_exploded.csv'), index=False)
df_genres.to_csv(os.path.join(processed_dir, 'netflix_genres_exploded.csv'), index=False)

print("Datasets exportados com sucesso!")

check = pd.read_csv(os.path.join(processed_dir, 'netflix_genres_exploded.csv'))
print(f"\n[Validação pós-exportação]")
print(f"Linhas no CSV salvo: {len(check)} (esperado: {len(df_genres)})")
assert len(check) == len(df_genres), "O CSV salvo não bate com o df_genres em memória - reexportação falhou"
print("Confirmado: o arquivo em disco corresponde à versão corrigida em memória.")

Datasets exportados com sucesso!

[Validação pós-exportação]
Linhas no CSV salvo: 19323 (esperado: 19323)
Confirmado: o arquivo em disco corresponde à versão corrigida em memória.


### 5. Nota de Depuração — Explode de Gêneros
- **Achado:** A primeira tentativa de explode em `listed_in` gerou 263.791 linhas (vs. ~19.000 esperadas),
  causado por uma transformação incorreta que quebrou as strings caractere por caractere em vez de
  separá-las por vírgula.
- **Correção:** Reescrita da célula usando `.str.split(',')` explicitamente.
- **Validação:** 19.323 linhas, 42 gêneros únicos, média de 2,19 gêneros/título — consistente com o
  esperado para este dataset.